# 📌 Tải & Khởi Tạo Dữ Liệu Dự Án (Datasets Downloader)

Notebook này thực hiện việc tự động tải các bộ dữ liệu từ Kaggle và lưu trữ vào thư mục dữ liệu thô (`data/raw/`).

---

### 🛠️ Cơ chế hoạt động:
* **D1 (Hotel Booking Demand)** & **D2 (Instacart Market Basket)**: Tải và lưu giữ nguyên vẹn 100% dữ liệu gốc.
* **D3 (US Accidents)**: Do file gốc quá lớn (~2.85 GB), script áp dụng kỹ thuật **Chunking** để:
  1. Đọc dữ liệu theo từng khối (100,000 dòng/lần) để tránh tràn RAM.
  2. Trích xuất **200,000 bản ghi mẫu** ngẫu nhiên (`random_state=42`).
  3. Tự động xóa file 2.85 GB gốc trong bộ nhớ đệm (cache) ngay sau khi tạo file sample (~40 MB).

---

### 📂 Cấu trúc dữ liệu đầu ra:
* `data/raw/D1_hotel_booking/hotel_bookings.csv`
* `data/raw/D2_instacart/` *(chứa các file CSV liên quan)*
* `data/raw/D3_us_accidents/US_Accidents_sampled.csv`

---
*▶️ **Hướng dẫn:** Chạy toàn bộ file để tiến hành tải và khởi tạo dữ liệu.*

In [1]:
import os
import sys
import subprocess
import shutil

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
RAW_DIR = os.environ.get("RAW_DATA_DIR", SCRIPT_DIR)
os.makedirs(RAW_DIR, exist_ok=True)


def ensure_package(package: str, import_name: str = None) -> None:
    """Tự động cài đặt package bằng pip nếu chưa có sẵn."""
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        print(f"📦 Chưa có '{package}', đang cài đặt...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )
        print(f"✅ Đã cài xong '{package}'")


# Đảm bảo có sẵn kagglehub và pandas
ensure_package("kagglehub")
ensure_package("pandas")


📦 Chưa có 'kagglehub', đang cài đặt...
✅ Đã cài xong 'kagglehub'


In [2]:
import kagglehub
import pandas as pd

DATASETS = {
    "D1_hotel_booking": "jessemostipak/hotel-booking-demand",
    "D2_instacart": "psparks/instacart-market-basket-analysis",
    "D3_us_accidents": "sobhanmoosavi/us-accidents",
}

def process_d3_sampling(cache_path: str, dest_dir: str, target_rows: int = 200000, random_state: int = 42) -> None:
    """
    Xử lý riêng cho D3: Đọc theo chunk, lấy mẫu ngẫu nhiên và xóa file gốc 2.85GB trong cache.
    """
    os.makedirs(dest_dir, exist_ok=True)

    # Tìm file CSV lớn trong thư mục cache
    csv_file = None
    for file_name in os.listdir(cache_path):
        if file_name.endswith(".csv"):
            csv_file = os.path.join(cache_path, file_name)
            break

    if not csv_file:
        raise FileNotFoundError("Không tìm thấy file CSV của D3 trong cache!")

    print(f"⚡ Đang đọc và lấy mẫu {target_rows:,} bản ghi từ file gốc D3 (dùng Chunking)...")

    # Đọc theo chunk 100,000 dòng để không làm tràn RAM
    chunk_size = 100000
    sampled_chunks = []

    for chunk in pd.read_csv(csv_file, chunksize=chunk_size, low_memory=False):
        # Lấy khoảng 5% từ mỗi chunk
        s = chunk.sample(frac=0.05, random_state=random_state)
        sampled_chunks.append(s)

        # Ngừng khi đã thu đủ hoặc vượt số lượng mẫu mong muốn
        if sum(len(c) for c in sampled_chunks) >= target_rows:
            break

    # Gộp các chunk đã lấy mẫu và cắt đúng target_rows
    df_sample = pd.concat(sampled_chunks, axis=0).head(target_rows)

    # Lưu ra file CSV mẫu nhẹ hơn nhiều trong thư mục raw
    output_file = os.path.join(dest_dir, "US_Accidents_sampled.csv")
    df_sample.to_csv(output_file, index=False)

    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"✅ Đã tạo thành công file sample: {output_file} ({size_mb:.2f} MB)")

    # 🧹 DỌN DẸP CACHE: Xóa sạch thư mục cache chứa file 2.85GB gốc
    print("🧹 Đang xóa file 2.85GB gốc khỏi bộ nhớ đệm cache...")
    shutil.rmtree(cache_path, ignore_errors=True)
    print("✨ Dọn dẹp cache hoàn tất!")

def download_all(datasets: dict) -> dict:
    """
    Tải toàn bộ dataset trong `datasets`:
    - D1, D2: Copy nguyên vẹn toàn bộ file.
    - D3: Đọc chunk -> Lấy mẫu -> Lưu file nhẹ -> Xóa cache gốc.
    """
    final_paths = {}

    for name, dataset_id in datasets.items():
        dest_dir = os.path.join(RAW_DIR, name)

        # Nếu thư mục đích đã có dữ liệu thì bỏ qua
        if os.path.isdir(dest_dir) and os.listdir(dest_dir):
            print(f"⏭️  Bỏ qua {name}: đã tồn tại tại {dest_dir}")
            final_paths[name] = dest_dir
            print("-" * 60)
            continue

        print(f"⬇️  Đang tải {name} ({dataset_id}) ...")
        try:
            # kagglehub tải/caching về ~/.cache/kagglehub/
            cache_path = kagglehub.dataset_download(dataset_id)

            if name == "D3_us_accidents":
                # Xử lý lấy mẫu riêng cho D3
                process_d3_sampling(cache_path, dest_dir, target_rows=200000)
            else:
                # D1 và D2 copy nguyên vẹn như ban đầu
                shutil.copytree(cache_path, dest_dir, dirs_exist_ok=True)

            final_paths[name] = dest_dir
            print(f"✅ Xong: {name} -> {dest_dir}")
        except Exception as e:
            print(f"❌ Lỗi khi tải {name}: {e}")
        print("-" * 60)

    return final_paths


def list_dataset_files(downloaded_paths: dict) -> None:
    """In ra danh sách file và dung lượng của từng dataset đã tải."""
    for name, path in downloaded_paths.items():
        print(f"\n📁 {name}  ({path})")
        if not os.path.isdir(path):
            print("   (không tìm thấy thư mục)")
            continue
        for f in sorted(os.listdir(path)):
            full_path = os.path.join(path, f)
            if os.path.isfile(full_path):
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                print(f"   - {f} ({size_mb:.2f} MB)")
            else:
                print(f"   - {f}/ (thư mục con)")


def main():
    print("=" * 60)
    print("TẢI DATASET TỪ KAGGLE (CÓ SAMPLING CHO D3)")
    print(f"Thư mục lưu trữ: {RAW_DIR}")
    print("=" * 60)

    downloaded_paths = download_all(DATASETS)

    print("\n📂 Tổng hợp đường dẫn:")
    for name, path in downloaded_paths.items():
        print(f"   {name}: {path}")

    print("\n🔍 Chi tiết nội dung từng dataset:")
    list_dataset_files(downloaded_paths)

    return downloaded_paths


if __name__ == "__main__":
    paths = main()


TẢI DATASET TỪ KAGGLE (CÓ SAMPLING CHO D3)
Thư mục lưu trữ: e:\CODECNTT\DU_AN\DataMining\data-mining\data\raw
⬇️  Đang tải D1_hotel_booking (jessemostipak/hotel-booking-demand) ...
❌ Lỗi khi tải D1_hotel_booking: [WinError 3] The system cannot find the path specified: 'C:\\Users\\nguye\\.cache\\kagglehub'
------------------------------------------------------------
⬇️  Đang tải D2_instacart (psparks/instacart-market-basket-analysis) ...
❌ Lỗi khi tải D2_instacart: [WinError 3] The system cannot find the path specified: 'C:\\Users\\nguye\\.cache\\kagglehub'
------------------------------------------------------------
⬇️  Đang tải D3_us_accidents (sobhanmoosavi/us-accidents) ...
❌ Lỗi khi tải D3_us_accidents: [WinError 3] The system cannot find the path specified: 'C:\\Users\\nguye\\.cache\\kagglehub'
------------------------------------------------------------

📂 Tổng hợp đường dẫn:

🔍 Chi tiết nội dung từng dataset:
